<a href="https://colab.research.google.com/github/iantonyjohn/Map-making/blob/main/Map_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import necessary packages

In [ ]:
!pip install geopandas matplotlib contextily shapely matplotlib-scalebar tqdm

In [ ]:
import json
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import matplotlib.patheffects as pe
from shapely.geometry import shape
from matplotlib.patches import FancyArrowPatch, Patch
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar

def plot_geojson_on_basemap(
    geojson_string,
    output_file="plot_map_final.png",
    pad_factor=0.30,
    figsize=(16.54, 11.69),  # A3 Landscape
    dpi=300
):
    """
    Professional map visualization for A3 print with transparent attribution.
    """
    # 1. Parse GeoJSON
    data = json.loads(geojson_string)
    gdf = gpd.GeoDataFrame.from_features(data if data.get('type') == 'FeatureCollection' else [data], crs="EPSG:4326")

    # 2. Reproject
    utm_crs = gdf.estimate_utm_crs()
    gdf_utm = gdf.to_crs(utm_crs)
    gdf_web = gdf.to_crs(epsg=3857)

    # 3. Stats Calculation
    primary_geom = gdf_utm.geometry.iloc[0]
    if primary_geom.geom_type in ['Polygon', 'MultiPolygon']:
        area_ha = primary_geom.area / 10000.0
        info_text = f"Area: {area_ha:.2f} ha"
    else:
        info_text = "Point Geometry Plot"

    # 4. Extent
    minx, miny, maxx, maxy = gdf_web.total_bounds
    w, h = maxx - minx, maxy - miny
    pad_x, pad_y = max(w, 100) * pad_factor, max(h, 100) * pad_factor

    # 5. Plotting
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # Split Geometries
    polys = gdf_web[gdf_web.geometry.type.isin(['Polygon', 'MultiPolygon'])]
    points = gdf_web[gdf_web.geometry.type.isin(['Point', 'MultiPoint'])]

    # Plot Geometries
    if not polys.empty:
        polys.boundary.plot(ax=ax, edgecolor="#00FFFF", linewidth=2.5, zorder=5)
        polys.plot(ax=ax, facecolor="#00FFFF", alpha=0.15, zorder=4)
    if not points.empty:
        points.plot(ax=ax, color="red", markersize=80, edgecolor="white", linewidth=1.2, zorder=10)

    # Set Map View
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    # 6. Satellite Basemap
    ctx.add_basemap(ax, source="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
                    crs=gdf_web.crs.to_string(), attribution=False, zorder=1)

    # 7. Legend
    legend_elements = [
        Patch(facecolor='#00FFFF', edgecolor='#00FFFF', alpha=0.3, label='Farm boundary'),
        Line2D([0], [0], marker='o', color='w', label='Sampling points',
               markerfacecolor='red', markersize=10, markeredgecolor='white')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=12,
              framealpha=0.9, facecolor='white', edgecolor='black')

    # 8. Styling & Text Labels
    ax.set_axis_off()

    # North Arrow
    north_arrow = FancyArrowPatch((0.95, 0.88), (0.95, 0.94), transform=ax.transAxes,
                                  mutation_scale=20, facecolor="white", edgecolor="black", zorder=15)
    ax.add_patch(north_arrow)
    ax.text(0.95, 0.95, "N", transform=ax.transAxes, ha="center", weight="bold",
            color="white", path_effects=[pe.withStroke(linewidth=2, foreground="black")], zorder=16)

    # Scale Bar
    ax.add_artist(ScaleBar(dx=1, units="m", location="lower left", box_alpha=0.7))

    # Repositioned Area Tag
    ax.text(0.02, 0.96, info_text, transform=ax.transAxes, fontsize=11, weight='bold',
            color="black", bbox=dict(facecolor="white", alpha=0.8, boxstyle="round"), zorder=12)

    # Transparent Watermark and Disclaimer
    attribution = "Map attribution: Antony John, Spatial data analyst, INDOCERT"
    ax.text(0.5, 0.02, f"{attribution}\nmap is not to scale", transform=ax.transAxes,
            fontsize=8, ha="center", color="white", alpha=0.05,
            path_effects=[pe.withStroke(linewidth=0.5, foreground="black", alpha=0.05)], zorder=20)

    plt.tight_layout()
    plt.savefig(output_file, dpi=dpi, bbox_inches='tight')
    plt.close(fig)

In [ ]:
import os
from tqdm.auto import tqdm

input_folder = input("Enter the path to the folder containing GeoJSON files: ")
output_folder = "output_maps_print"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

if os.path.isdir(input_folder):
    files_found = [f for f in os.listdir(input_folder) if f.lower().endswith(('.json', '.geojson'))]

    if not files_found:
        print(f"No valid GeoJSON files found in {input_folder}")
    else:
        # Added tqdm progress bar to the loop
        for filename in tqdm(files_found, desc="Generating Maps"):
            try:
                file_path = os.path.join(input_folder, filename)
                with open(file_path, 'r') as f:
                    data = f.read()

                out_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".png")
                plot_geojson_on_basemap(geojson_string=data, output_file=out_path)
            except Exception as e:
                print(f"\nFailed to process {filename}: {e}")

    print(f"\nDone! All maps saved in '{output_folder}' at 300 DPI.")
else:
    print("Invalid directory.")

Enter the path to the folder containing GeoJSON files: /content/


Generating Maps:   0%|          | 0/20 [00:00<?, ?it/s]


Done! All maps saved in 'output_maps_print' at 300 DPI.


In [ ]:
import shutil
from google.colab import files

# Name for the zip file
zip_filename = "output_maps_print_collection"

# Create the zip archive
shutil.make_archive(zip_filename, 'zip', "output_maps_print")

# Trigger the download
files.download(f"{zip_filename}.zip")

print(f"Success: '{zip_filename}.zip' has been created and the download has started.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Success: 'output_maps_print_collection.zip' has been created and the download has started.
